# Permutation Importance

Which survey variables does the tuned random forest rely on for each mental-health target?
Computed out of fold on the training set only.

- For each target, max_features and min_samples_leaf are tuned once on the whole training set
  with the inner CV.
- The training set is split into 5 folds, repeated twice (10 held-out folds). In each fold a
  forest with the tuned settings is fitted on 4/5 of the data, and each survey variable is
  shuffled 10 times in the held-out fifth. The importance is the mean drop in R².
- Each survey variable is shuffled as a whole, so Fav genre and Primary streaming service get
  one importance each, not one per category.
- The table reports the mean drop across the 10 folds with the corrected 95% CI. A variable
  whose CI includes zero has no reliable importance. Read the results alongside the low overall
  R² from the nested CV.
- The settings were tuned on the whole training set, so the held-out folds are not fully
  independent of tuning. This matters little for ranking the variables.

In [ ]:
import pandas as pd

import mxmh_common as mc

# Set RERUN = False to use the saved per-fold results if they exist.
RERUN = True
FOLDS_PATH = mc.RESULTS_DIR / "permutation_importance_folds.csv"

if RERUN or not FOLDS_PATH.exists():
    folds = mc.run_permutation_importance()
else:
    folds = pd.read_csv(FOLDS_PATH)
    print(f"Loaded {len(folds)} fold results from {FOLDS_PATH}")

# Training and held-out sizes of each fold, needed for the corrected CI
X, Y = mc.load_clean_dataset()
X_train, _, _, _ = mc.split_train_test(X, Y)
n_train, n_test = mc.outer_fold_sizes(len(X_train))

## Importance by Variable

Mean drop in R² when the variable is shuffled, with the corrected 95% CI. "Positive folds" is
the share of the 10 folds where shuffling lowered R². Variables are sorted by mean drop within
each target; the full table is saved, and the top 8 per target are shown.

In [ ]:
TOP_N = 8

rows = []
for (target, variable), group in folds.groupby(["target", "variable"]):
    ci = mc.corrected_ci(group.importance, n_train, n_test)
    rows.append({"Target": target, "Variable": variable, "mean": ci["mean"],
                 "Drop in R2": f"{ci['mean']:.3f} [{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]",
                 "Positive folds": f"{ci['share_positive']:.0%}"})
importance = (pd.DataFrame(rows)
              .sort_values(["Target", "mean"], ascending=[True, False])
              .drop(columns="mean"))
importance.to_csv(mc.RESULTS_DIR / "permutation_importance_summary.csv", index=False)
importance.groupby("Target").head(TOP_N).set_index(["Target", "Variable"])